# 06 - Markov models and Dynamic HAZOP

## Learning objectives

- Distinguish DTMCs and CTMCs
- Construct and validate a generator matrix
- Evaluate a failure-repair CTMC
- Connect TK-101 deviations to an illustrative state model


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. What is missing from static FTA?

A static fault tree does not naturally describe event order, time spent degraded or repair dynamics. A Markov model represents temporal behavior through **states** and **transition rates or probabilities**.


## 2. DTMC

Two states: OK and FAILED. For one step, $P=\begin{bmatrix}.98&.02\\.20&.80\end{bmatrix}$. Every row sums to one.


In [ ]:
P=np.array([[.98,.02],[.20,.80]])
p=np.array([1.,0.]); hist=[p.copy()]
for _ in range(20): p=p@P; hist.append(p.copy())
hist=np.array(hist)
plt.plot(hist[:,0],label='OK'); plt.plot(hist[:,1],label='FAILED'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 3. CTMC and generator matrix

Two-state failure-repair model:

$$Q=\begin{bmatrix}-\lambda&\lambda\\\mu&-\mu\end{bmatrix},\qquad p(t)=p(0)e^{Qt}.$$

Off-diagonal entries are nonnegative; every row sums to zero.


In [ ]:
from scipy.linalg import expm
lam=.002; mu=.08
Q=np.array([[-lam,lam],[mu,-mu]])
p0=np.array([1.,0.]); times=np.linspace(0,200,200)
probs=np.vstack([p0@expm(Q*t) for t in times])
plt.plot(times,probs[:,0],label='OK'); plt.plot(times,probs[:,1],label='FAILED'); plt.axhline(mu/(lam+mu),ls='--',label='A∞'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 4. TK-101 teaching states

S0 Normal; S1 Sensor degraded; S2 Protection degraded/maintenance overdue; S3 High-level challenge; S4 Overflow; S5 Fire.

This compact model does not separate every barrier combination. Once trajectories enter S3, their previous degradation state is not retained. Therefore it is an illustrative aggregation, not a validated barrier-dependent process model.


In [ ]:
states=['S0_normal','S1_sensor_degraded','S2_protection_degraded','S3_challenge','S4_overflow','S5_fire']; n=len(states)
Q=np.zeros((n,n))
def rate(i,j,x): Q[i,j]+=x
rate(0,1,.002); rate(0,2,.001); rate(0,3,.004)
rate(1,0,.08); rate(1,3,.004); rate(2,0,.05); rate(2,3,.004)
rate(3,0,.20); rate(3,4,.01); rate(4,0,.10); rate(4,5,.03)
for i in range(n): Q[i,i]=-Q[i].sum()
pd.DataFrame(Q,index=states,columns=states)


In [ ]:
p0=np.array([1,0,0,0,0,0.],float); ts=np.linspace(0,300,301); P=np.vstack([p0@expm(Q*t) for t in ts])
for j,s in enumerate(states): plt.plot(ts,P[:,j],label=s)
plt.xlabel('h'); plt.ylabel('state probability'); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()


## 5. Dynamic HAZOP illustration

Interpret `MORE LEVEL` as a temporal path: **normal/degraded -> high-level challenge -> overflow -> fire**. Safeguards or maintenance can provide recovery transitions.

This links deviations to state evolution. It is not a complete dynamic process simulation or a substitute for a reviewed engineering HAZOP study.


In [ ]:
pd.DataFrame([
 {'deviation':'MORE LEVEL','from':'S0/S1/S2','to':'S3','barrier':'alarm/trip/operator'},
 {'deviation':'NO EFFECTIVE SHUTDOWN','from':'S3','to':'S4','barrier':'LSHH+XV+manual recovery'},
 {'deviation':'IGNITION AFTER RELEASE','from':'S4','to':'S5','barrier':'containment+fire protection'}])


## 6. Maintenance sensitivity

Vary the sensor restoration rate and compute the probability of occupying overflow or fire **at 100 hours**. This is not the probability of having experienced overflow at any time within the first 100 hours; a first-passage calculation would require an appropriate absorbing-state construction.


In [ ]:
def with_repair(r):
    q=Q.copy(); q[1,0]=r; q[1,1]=-q[1,np.arange(n)!=1].sum(); return q
rows=[]
for r in [.02,.05,.08,.15]:
    p100=p0@expm(with_repair(r)*100); rows.append({'repair_rate':r,'P_S4_or_S5_100h':p100[4]+p100[5]})
pd.DataFrame(rows)


In [ ]:
from safetycourse.markov import validate_generator,transient_probabilities
validate_generator(Q); transient_probabilities(Q,p0,[0,10,100])


## 7. Limitations

The Markov property makes the future depend on the current modeled state. If age, history or time spent degraded matters, augment the state or choose another model. Exponential holding times are not suitable for every process. Check whether state aggregation preserves the dependencies needed for the engineering question.


## Try it!

1. Increase the S3 -> S4 rate to 0.03.
2. Reduce S3 -> S0 recovery to 0.05.
3. Add S5 -> S0 repair and justify the physical interpretation.
4. Add a combined S1+S2 state and inspect state-space growth.
5. Separate S3 according to protection availability before using this model to infer maintenance benefits.


## Summary

Markov models describe time-dependent state probabilities. The Dynamic HAZOP illustration links hazardous deviations to this state representation. Next, examine uncertainty and rare events using simulation.
